In [0]:
bronze_layer_path = "/Volumes/workspace/default/bronze_layer"

bronze_layer = {}

folders = dbutils.fs.ls(bronze_layer_path)

for folder in folders:

    if folder.isDir():

        df = (
            spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .load(folder.path)
        )

        table_name = folder.name.removesuffix(".csv")

        bronze_layer[table_name] = df

        print(f"File loaded successfully: {table_name}")

In [0]:
for table_name,df in bronze_layer.items():
    print(f"table_name{table_name}")
    df.display(5,truncate=False)

In [0]:
bronze_layer_path = "/Volumes/workspace/default/bronze_layer"

bronze_layer = {}

folders = dbutils.fs.ls(bronze_layer_path)

for folder in folders:

    if folder.isDir():

        folder_path = folder.path
        table_name = folder.name.rstrip("/")

        contents = dbutils.fs.ls(folder_path)

        # Check source format
        is_delta = any(
            item.name == "_delta_log/"
            for item in contents
        )

        if is_delta:

            df = (
                spark.read
                .format("delta")
                .load(folder_path)
            )

        else:

            csv_files = [
                item.path
                for item in contents
                if item.name.lower().endswith(".csv")
            ]

            if not csv_files:
                continue

            df = (
                spark.read
                .format("csv")
                .option("header", "true")
                .option("inferSchema", "true")
                .load(csv_files)
            )

        bronze_layer[table_name] = df

        print(f"Successfully extracted: {table_name}")

In [0]:
files = dbutils.fs.ls(bronze_layer_path)

for file in files:
    print(file.name, file.path)

In [0]:
for table_name, df in bronze_layer.items():
    print(f"Dataset: {table_name}")
    df.display(5,truncate = False)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col,
    row_number
)

# Define primary keys for each dataset
primary_keys = {
    "customers/": "customer_id",
    "orders/": "order_id",
    "order_items/": "order_item_id",
    "product/": "product_id"
}

silver_layer = {}

for table_name, df in bronze_layer.items():

    primary_key = primary_keys[table_name]

    window_spec = (
        Window
        .partitionBy(primary_key)
        .orderBy(col("updated_at").desc())
    )

    deduplicated_df = (
        df
        .withColumn("row_num", row_number().over(window_spec))
        .filter(col("row_num") == 1)
        .drop("row_num")
    )

    silver_layer[table_name] = deduplicated_df

    print(f"Deduplication completed: {table_name}")

In [0]:
table_name = folder.name.rstrip("/")
primary_key = primary_keys[table_name]

In [0]:
for table_name, df in bronze_layer.items():

    table_name = table_name.rstrip("/")

    primary_key = primary_keys[table_name]

In [0]:
print("Bronze dictionary keys:")

for key in bronze_layer.keys():
    print(repr(key))

print("\nPrimary key dictionary:")

for key in primary_keys.keys():
    print(repr(key))

In [0]:
table_name = table_name.rstrip("/")
primary_key = primary_keys[table_name]

In [0]:
customers/ dbfs:/Volumes/workspace/default/bronze_layer/customers/
order_items/ dbfs:/Volumes/workspace/default/bronze_layer/order_items/
orders/ dbfs:/Volumes/workspace/default/bronze_layer/orders/
product/ dbfs:/Volumes/workspace/default/bronze_layer/product/

In [0]:
# Clean Bronze dictionary keys
bronze_layer = {
    key.rstrip("/"): df
    for key, df in bronze_layer.items()
}

# Clean primary key dictionary keys
primary_keys = {
    key.rstrip("/"): value
    for key, value in primary_keys.items()
}

print("Bronze keys:", list(bronze_layer.keys()))
print("Primary keys:", list(primary_keys.keys()))

In [0]:
for table_name, df in bronze_layer.items():
    print(f"\nDataset: {table_name}")
    print(df.columns)